# Day 4 — Neural Networks with PyTorch (MT2026)

**Minimum route:** run the independent batch/logits example, complete the loader and MLP layer exercises, and observe one forward/loss/update step. **Expected route:** run a capped training demo and explain generalisation. **Extension:** complete the unchanged full activation, CNN, regularisation and early-stopping exercises.

### How to use this practical

- **CORE EXERCISE 1/2–2/2:** two priority coding tasks; find **Task 1** and **Task 2** in the exercise descriptions and code cells.
- **PROVIDED EXAMPLE:** working code to run, inspect and discuss.
- **OPTIONAL EXERCISE:** further experiments and implementations (Tasks 3–6).

**Classroom pacing:** the complete CIFAR-10 experiments can take much longer than two hours on a CPU. Running all epochs is not required to complete the core. The small independent demonstration below works even before downloading the dataset.

**Optional training on a CPU:** start with `num_epochs = 3` to `5` for a quick trial; full experiments may take much longer. A Colab GPU can be used for longer runs.


### PROVIDED EXAMPLE

Batch and logits without downloads.

The shape `(4,3,32,32)` means 4 colour images. A classifier for 10 categories should output `(4,10)` unnormalised class scores, called **logits**.

In [6]:
import torch
import torch.nn as nn
example_images = torch.randn(4, 3, 32, 32)
example_logits = torch.randn(4, 10)
print('image batch:', example_images.shape, 'logits:', example_logits.shape)

image batch: torch.Size([4, 3, 32, 32]) logits: torch.Size([4, 10])


# Introduction to Neural Networks in PyTorch by `Qurat ul ain`
In this practical, you will build a simple neural network to classify images and experiment with various neural network components, including activation functions, regularization, and optimization methods.

### Objectives
- Build a feedforward neural network for image classification.
- Experiment with different activation functions and observe their effects.
- Apply regularization techniques to prevent overfitting.
- Learn model checkpointing and Early stopping

### Required packages:

[1] [pytorch](https://pytorch.org/docs/stable/index.html) is library widely used for bulding deep-learning frameworks

[2] [TorchVision](https://pytorch.org/vision/stable/index.html) you can use available datasets that include one you are required to use in this tutorial (CIFAR10) and others like MNIST

[3] [matplotlib](http://matplotlib.org) can be used for plotting graphs in python


In [7]:
from IPython.display import clear_output

**Setup:** Install packages once using the shared `requirements.txt` and Setup Instructions before running the notebook.

In [8]:
# Dependencies are installed separately with requirements.txt.
# See the course Setup Instructions before running the notebook.

In [9]:
# Import necessary libraries

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import time
from torchvision.datasets import CIFAR10,MNIST
from torch.utils.data import DataLoader

from torchvision.transforms.functional import to_tensor, to_pil_image, resize
device = 'cuda' if torch.cuda.is_available() else 'cpu'  # checks if machine supports cuda and if it does, we use that, otherwise cpu

print(device)#should say cuda if you are on a GPU

cpu


## Step 1: Load and Preprocess the Data


CIFAR10:

The CIFAR-10 dataset consists of 60000 32x32 colour images (so 3 channels) in 10 classes, with 6000 images per class. There are 50000 training images and 10000 test images.

Example CIFAR-10 images can be viewed in the image-visualisation cell below.


### Instructions
1. Load the dataset using `torchvision.datasets`.
2. Use `DataLoader` to batch the data. Batch size refers to the number of training examples utilized in one iteration

Without a transform, CIFAR-10 returns PIL images. In this notebook, `transform=to_tensor` converts them to floating-point tensors with pixel values in [0, 1].


- Transform to Tensors:
https://pytorch.org/vision/stable/transforms.html#torchvision.transforms.ToTensor

**Data preprocessing: <br>**
*Step [1]:* Transform --> Always convert data to ToTensor (you can do **step 1, 2, and 3** in one line as done in this tutorial)

*Step [2]:* Make [DataLoaders](https://pytorch.org/docs/stable/data.html): It represents a Python iterable over a dataset

In [10]:
dataset_root = 'data/'

# Download from the course dataset link only when CIFAR-10 is not already present.
from prepare_cifar10_from_drive import ensure_cifar10
ensure_cifar10(dataset_root)

train_dataset = CIFAR10(root=dataset_root, train=True, download=False, transform=to_tensor)
test_dataset = CIFAR10(root=dataset_root, train=False, download=False, transform=to_tensor)


print('Length of train_dataset is', len(train_dataset))
print('Length of val_dataset is'  , len(test_dataset))


RuntimeError: Install the course requirements first: python -m pip install -r requirements.txt

### CORE EXERCISE 1/2

**Task 1:** Create the evaluation `DataLoader` with `shuffle=False`.

**solution**

In [ ]:
# Create DataLoaders
batch_size = 100

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)  # no need to shuffle validation data


## Visualize Sample Data
Let's display a few sample images to understand the dataset.


In [ ]:
# Get a random training images
random_img_idx = torch.randint(0, 1000, (1,)).item()


test_image = train_dataset[random_img_idx][0]  # 0 for image part in (image, label) tuple.
print('The size of image variable is Channel, Height, Width ',test_image.shape)

test_image = resize(test_image, (250, 250), antialias=None)  # better visualization
print('Number of channels in test_image: ', test_image.shape[0])
to_pil_image(test_image)

## Step 2: Define a Simple Neural Network
We'll start by defining a simple feedforward neural network with two hidden layers.

`nn.Linear` module automatically creates a linear transformation with bias and weight tensors that can be used in the forward feed. After defining the layers, you can access these parameters using `model.fc1.weight` or `model.fc1.bias`, for example.


Remember the size of an image in CIFAR is 3x32x32. Based on that think carefuly about what the input dimension of your first layer should be.

Remember the total number of unique classes (images) in CIFAR is 10. Based on that think carefuly about what the output dimension of your last layer should be.



**solution — input/output dimensions**

CIFAR-10 inputs are RGB images of shape `(3,32,32)`, so flattening gives $3\times32\times32=3072$ features per image. Ten mutually exclusive categories require 10 output logits, not a single output neuron. `nn.CrossEntropyLoss()` expects those logits and integer class indices.

### PROVIDED EXAMPLE

A short loss and update demonstration, independent of CIFAR-10 downloads.

Predict four random image labels, calculate cross-entropy from **raw logits**, then perform one update. Watch which values are parameters, gradients and predictions.

In [ ]:
toy_model = nn.Sequential(nn.Flatten(), nn.Linear(3*32*32, 10))
toy_x = torch.randn(4, 3, 32, 32)
toy_y = torch.tensor([0, 1, 2, 3])
toy_opt = torch.optim.SGD(toy_model.parameters(), lr=0.01)
toy_opt.zero_grad()
toy_logits = toy_model(toy_x)
toy_loss = nn.CrossEntropyLoss()(toy_logits, toy_y)
toy_loss.backward()
print('logits shape:', tuple(toy_logits.shape), 'loss:', round(toy_loss.item(), 3))
print('gradient shape:', tuple(toy_model[1].weight.grad.shape))
toy_opt.step()

### CORE EXERCISE 2/2

**Task 2:** Define the three `nn.Linear` layers for the MLP: flattened 3×32×32 inputs → 128 → 64 → 10 logits.

**solution**

In [ ]:
class SimpleNN(nn.Module):
    """
    A simple Multi-Layer Perceptron (MLP) model for image classification using fully connected layers.

    This neural network has three layers:
      - Input Layer (fc1): Takes a flattened input image and maps it to 128 neurons.
      - Hidden Layer (fc2): Maps the output of the first layer to 64 neurons.
      - Output Layer (fc3): Maps the 64 neurons to 10 output neurons.

    Parameters:
    -----------
    activation_function : nn.Module
        The activation function to be applied after each fully connected layer (e.g., ReLU, Sigmoid, Tanh).

    Methods:
    --------
    forward(x):
        Defines the forward pass of the network. Applies the activation function after each layer, except the final layer.

    Example:
    --------
    model = SimpleNN(activation_function=nn.ReLU())
    output = model(input_image)

    Notes:
    ------
    - `x.view(-1, 3*32*32)` reshapes the input images from (batch_size, 3,32,32) to (batch_size, 3072) for the fully connected layer.
    - Each activation function transforms layer outputs, introducing non-linearity and enabling the network to learn complex patterns.
    """

    def __init__(self, activation_function):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(3*32*32, 128)  # Input layer: Flatten 3×32×32 to 3072 and map to 128
        self.fc2 = nn.Linear(128, 64)     # Hidden layer: Map 128 to 64
        self.fc3 = nn.Linear(64, 10)      # Output layer: Map 64 to 10

        # Store the activation function
        self.activation_function = activation_function

    def forward(self, x):
        # Flatten the input tensor
        x = x.view(-1, 3*32*32)
        x = self.activation_function(self.fc1(x))
        x = self.activation_function(self.fc2(x))
        x = self.fc3(x)
        return x
SimpleNN(nn.ReLU())

## Define Training and Evaluation Functions
Let's define the evaluation functions first because it's easy and will be called in training function later.
We will define these function such that they return accuracy which we can plot later on. Ideally you should plot both loss and accuracy per epoch and keep a close eye on both.

- CAUTION: In this notebook, we use test_loader directly in evaluate_model just to make things simpler. In real life, if test_loader represents your final test dataset, this should not be used during training at all.
Instead, a separate validation set split from the training data should be used for the validation phase during training. This avoids data leakage into the final evaluation.
    

### PROVIDED EXAMPLE

Evaluation loop: inspect how predictions and accuracy are calculated.

In [ ]:
# Evaluation function
def evaluate_model(model, criterion, test_loader,device):
    """
    Evaluates the performance of a trained neural network model on a validation dataset.

    This function computes the accuracy of the model by comparing its predictions to the actual labels
    in the validation set. The accuracy metric is printed as a percentage of correctly classified samples.

    Parameters:
    -----------
    model : nn.Module
        The trained neural network model to be evaluated.
    criterion : nn.Module
        Loss function to use (e.g., CrossEntropyLoss).
    test_loader : DataLoader
        DataLoader for the validation set.

    Returns:
    --------
    val_loss : float
        Average validation loss over the dataset.
    val_accuracy : float
        Validation accuracy as a percentage.

    Process:
    --------
    - model.eval() Sets the model to evaluation mode ensuring that dropout and batch normalization behave properly during evaluation
    - Disables gradient computation with `torch.no_grad()` to speed up evaluation and save memory.
    - Iterates over each batch in the test DataLoader (`test_loader`):
        1. Passes each batch of images through the model to obtain predictions.
        2. Uses `torch.max` to find the predicted class label with the highest score.
        3. Compares the predicted labels to the actual labels and counts correct predictions.
    - Calculates accuracy as the percentage of correct predictions out of the total samples.

    Notes:
    ------
    - `torch.no_grad()` is used to reduce memory usage and computation, as gradients are not needed during evaluation.
    - `torch.max(outputs.data, 1)` retrieves the predicted class with the highest probability for each sample in the batch.
    - `correct / total * 100` gives the model's accuracy as a percentage, which is a commonly used metric for classification tasks.

    Example:
    --------
    evaluate_model(model)

    This will output the accuracy and loss of the model on the validation set, for example:
    """
    model.eval()  # Set the model to evaluation mode
    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute validation loss
            val_running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)  # Get predicted class
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    # Calculate validation metrics
    val_loss = val_running_loss / len(test_loader)
    val_accuracy = 100 * val_correct / val_total
    return val_loss, val_accuracy


### PROVIDED EXAMPLE

Training loop: follow the forward pass, loss, backward pass and parameter updates.

In [ ]:
# Training function
def train_model(model,criterion, optimizer, train_loader, test_loader,device, epochs=5):
    """
    Trains a neural network model using the specified loss function and optimizer.

    This function performs a forward pass, computes the loss, and applies backpropagation
    to update the model's weights for each epoch. The running loss for each epoch is calculated
    and displayed to track the training progress.

    Parameters:
    -----------
    model : nn.Module
        The neural network model to be trained.

    criterion : nn.Module
        The loss function used to measure how far the model's predictions are from the target labels.
        Common choices include CrossEntropyLoss for classification or MSELoss for regression.

    optimizer : torch.optim.Optimizer
        The optimization algorithm used to update model parameters based on gradients. Examples include
        SGD (Stochastic Gradient Descent) and Adam.

     train_loader : DataLoader
        DataLoader for the training set.

    val_loader : DataLoader
        DataLoader for the validation set.

    epochs : int, optional (default=5)
        The number of full passes through the training dataset.

    Returns:
    --------
    train_accuracies : list of floats
        Training accuracy for each epoch.

    val_accuracies : list of floats
        Validation accuracy for each epoch.

    Process:
    --------
    - For each epoch, reset the running loss.
    - For each mini-batch:
        1. Perform a forward pass to compute the model's predictions.
        2. Calculate the loss by comparing predictions with actual labels.
        3. Perform backpropagation to compute gradients.
        4. Use the optimizer to update the model's weights.
        5. Accumulate the batch loss into running_loss.
    - After each epoch, compute and print the average loss and accuracies.

    Notes:
    ------
    - `optimizer.zero_grad()` clears old gradients to prevent accumulation from previous batches.
    - `loss.backward()` computes gradients based on the current batch loss.
    - `optimizer.step()` updates the model's weights using the computed gradients.
    - `running_loss / len(train_loader)` provides the average loss per batch in each epoch,
      giving insight into how well the model is learning over time.


    Example:
    --------
    train_model(model, criterion=nn.CrossEntropyLoss(), optimizer=torch.optim.SGD(model.parameters(), lr=0.01), epochs=10)
    """
    train_accuracies = []
    val_accuracies = []

    for epoch in range(epochs):
        # Training phase
        model.train()  # Set the model to training mode
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()  # Clear previous gradients


            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            loss.backward()  # Backpropagation
            optimizer.step()  # Update weights

            running_loss += loss.item()  # Accumulate batch loss
            _, predicted = torch.max(outputs, 1)  # Get predicted class
            total += labels.size(0)  # Increment total samples
            correct += (predicted == labels).sum().item()  # Count correct predictions

        # Calculate epoch training accuracy
        train_accuracy = 100 * correct / total
        train_accuracies.append(train_accuracy)
        # Validation phase (call separate function)
        val_loss, val_accuracy = evaluate_model(model, criterion,test_loader,device) #CAUTION: See note above
        val_accuracies.append(val_accuracy)

        # Print progress
        print(
            f"Epoch [{epoch+1}/{epochs}], "
            f"Train Loss: {running_loss/len(train_loader):.4f}, Train Accuracy: {train_accuracy:.2f}%, "
            f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%"
        )

    return train_accuracies, val_accuracies

### PROVIDED EXAMPLE

Short-run alternative for the classroom.

The complete activation comparison immediately below uses three full trainings. You can instead run this small, capped comparison to explore the process; short-run accuracies are noisy and must not be interpreted as a fair ranking of activations.

In [ ]:
# Run this cell after the CIFAR-10 DataLoaders, SimpleNN and training helpers are ready.
# A small capped data subset keeps the demonstration manageable on CPU.
from torch.utils.data import Subset
small_train_loader = DataLoader(Subset(train_dataset, list(range(0, 256))), batch_size=64, shuffle=True)
small_eval_loader = DataLoader(Subset(test_dataset, list(range(0, 128))), batch_size=64, shuffle=False)
print('Quick-run batches:', len(small_train_loader), len(small_eval_loader))
# Try ONE activation with one epoch, then compare if you have time:
# model = SimpleNN(nn.ReLU()).to(device)
# optimizer = optim.SGD(model.parameters(), lr=0.01)
# train_model(model, nn.CrossEntropyLoss(), optimizer, small_train_loader, small_eval_loader, device, epochs=1)

## Step 3: Experimenting with Activation Functions
We'll test our network with different activation functions: ReLU, Sigmoid, and Tanh.

### Instructions
1. Instantiate the model with each activation function.
2. Train and evaluate the model with each activation function.
3. Observe and record your findings.


### OPTIONAL EXERCISE

**Task 3:** Run the ReLU, Sigmoid and Tanh comparison in Step 3, and compare their training and evaluation curves.

In [ ]:
num_epochs = 15
learning_rate = 0.001
print(f'Using device {device}')

In [ ]:
# List of activation functions to test
activations = [nn.ReLU(), nn.Sigmoid(), nn.Tanh()]
criterion = nn.CrossEntropyLoss()

for activation in activations:
    print(f"\nTesting with activation function: {activation}")
    model = SimpleNN(activation)
    model = model.to(device)
    optimizer = optim.SGD(model.parameters(), lr=learning_rate)

    # Start timer
    start_time = time.time()

    # Train the model
    train_accuracies, val_accuracies = train_model(model, criterion, optimizer, train_loader, test_loader, device,epochs=num_epochs)

    # Calculate elapsed time
    end_time = time.time()
    total_time = end_time - start_time

    # Print total time and accuracy
    print(f"Total time taken: {total_time:.2f} seconds")

        # Plot training and validation accuracy
    plt.figure(figsize=(10, 5))
    plt.plot(train_accuracies, label='Training Accuracy')
    plt.plot(val_accuracies, label='Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.title('{} Training and Validation Accuracy'.format(activation))
    plt.show()

#### Two take-aways from the above:
1. Compare the training and validation curves for ReLU, Tanh and Sigmoid. Which activation works best in your run?
2. This is a fully connected MLP with nonlinear activations, not a linear model. Does it generalise well to the evaluation data? Next, we will explore convolutional neural networks for CIFAR-10.



### Evaluating model fit
- **Underfitting**: Both training and validation losses are high and accuracy low, indicating that the model is not learning enough patterns from the data.
- **Good Fit**: Training and validation losses are both low and stable.
- **Overfitting**: Training loss continues to decrease, but validation loss does not improve or begins to increase, suggesting the model is memorizing the training data and is not generalizable.


## Step 4: Implementing CNN
We shall use ReLU as our activation function and Adam as our optimizer.

**solution — CNN feature dimensions**

With 32×32 input, stride-1 padded convolutions preserve 32×32. Each 2×2 max-pool halves spatial dimensions: 32→16→8. With 256 output channels after the final block, the flattened input to the first FC layer is `256 * 8 * 8 = 16384`. Print shapes in the forward pass when diagnosing an `nn.Linear` mismatch.

### OPTIONAL EXERCISE

**Task 4:** Complete the three fully connected layers and the forward pass in `SimpleCNN`.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # First Convolutional Block
        # Input: 3 channels (RGB), Output: 64 channels
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()


         # Second Convolutional Block
        # Input: 64 channels, Output: 128 channels
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        # Third Convolutional Block
        # Input: 128 channels, Output: 256 channels
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)

        # Pooling layer
        # Reduces spatial dimensions by half
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)


        # First fully connected layer
        # Input: Flattened feature maps (256 * 8 * 8), Output: 512 neurons
        self.flatten = nn.Flatten()  # Flatten layer for linear layers
        self.fc1 = nn.Linear(256 * 8 * 8, 512)  # Flattened size after pooling
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)  # Output size: 10 (CIFAR-10 classes)

    def forward(self, x):
        # Convolutional and pooling layers
        x = self.relu(self.conv1(x))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))

        # Flatten the tensor for the fully connected layers
        x = self.flatten(x)

        # Fully connected layers
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x
SimpleCNN()

In [ ]:
num_epochs = 25
learning_rate = 0.001

model = SimpleCNN().to(device)# send model to gpu
print(f'Using device {device}')
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


In [ ]:

# Start timer
start_time = time.time()

# Train the model
train_accuracies, val_accuracies = train_model(model, criterion, optimizer, train_loader, test_loader, device,epochs=num_epochs)

# Calculate elapsed time
end_time = time.time()
total_time = end_time - start_time

# Print total time and accuracy
print(f"Total time taken: {total_time:.2f} seconds")

    # Plot training and validation accuracy
plt.figure(figsize=(10, 5))
plt.plot(train_accuracies, label='Training Accuracy')
plt.plot(val_accuracies, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('CNN Training and Validation Accuracy')
plt.show()

We've overcorrected!!
Now, our model is a classic case of overfitting.
Let's add some regularization to keep training in check.


Btw, try switching Adam with SGD for the above block and see what happens ;)

## Step 5: Implementing Regularization

To prevent overfitting, implement the following:
- Apply Dropout after every fully connected (linear) layer (except the output layer)
- Apply Batch Normalization after every convolution layer.
- Apply L2 regularization (weight decay) to optimizer



**solution — regularisers**

Dropout sets a random subset of activations to zero during training and is disabled at evaluation. Batch normalisation stabilises intermediate activation statistics and uses running estimates during evaluation. Weight decay adds a parameter penalty (for standard SGD, closely related to L2 regularisation). Their effects should be assessed on validation data, not inferred from a predetermined accuracy number.

### OPTIONAL EXERCISE

**Task 5:** Complete `RegularizedCNN` using batch normalisation, ReLU, pooling, dropout and fully connected layers; then configure the optimiser with weight decay.

In [ ]:
class RegularizedCNN(nn.Module):
    """

    This model includes convolutional layers with Batch Normalization, ReLU activation, and MaxPooling,
    followed by fully connected layers with Dropout for regularization.

    Architecture:
    1. Three convolutional layers:
        - Convolution + Batch Normalization + ReLU
        - MaxPooling applied after the 2nd and 3rd convolutional layers
    2. Fully connected layers:
        - Includes Dropout
        - Outputs logits for 10 classes (e.g., CIFAR-10 classification)

    Methods:
    --------
    forward(x):
        Defines the forward pass through the network, applying the activation function, dropout, batch norm ,
        and fully connected layers. The output layer has no activation function applied.


    Notes:
    ------
    - Dropout is used as a regularization technique to reduce overfitting by randomly deactivating neurons during training..
    - No activation function is applied to the output layer, allowing it to output raw scores, which are used with a
      CrossEntropyLoss criterion.
    """

    def __init__(self):
        super(RegularizedCNN, self).__init__()

        # First Convolutional Block
        # Input: 3 channels (RGB), Output: 64 channels
        # BatchNorm normalizes the activations for stability
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()


         # Second Convolutional Block
        # Input: 64 channels, Output: 128 channels
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)

        # Third Convolutional Block
        # Input: 128 channels, Output: 256 channels
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(256)

        # Dropout for regularization
        self.dropout = nn.Dropout(0.5)  # Dropout with 50% probability

        # Pooling layer
        # Reduces spatial dimensions by half
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully connected layers
        # Flatten layer prepares feature maps for the linear layers
        self.flatten = nn.Flatten()

        # First fully connected layer
        # Input: Flattened feature maps (256 * 8 * 8), Output: 512 neurons
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(256 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        # Convolutional and pooling layers with BatchNorm
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))

        # Flatten for fully connected layers
        x = self.flatten(x)

        # Fully connected layers with BatchNorm and Dropout
        x = self.dropout(self.relu((self.fc1(x))))
        x = self.dropout(self.relu((self.fc2(x))))
        x = self.fc3(x)

        return x

RegularizedCNN()

**Task 5 (continued):** Add `weight_decay=1e-4` to the optimiser below.

In [ ]:
num_epochs = 40
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()

model = RegularizedCNN().to(device)# send model to gpu
print(f'Using device {device}')

optimizer = optim.Adam(model.parameters(), lr=learning_rate,weight_decay=1e-4)


In [ ]:
# Start timer
start_time = time.time()

# Train the model
train_accuracies, val_accuracies = train_model(model, criterion, optimizer, train_loader, test_loader, device,epochs=num_epochs)

# Calculate elapsed time
end_time = time.time()
total_time = end_time - start_time

# Print total time and accuracy
print(f"Total time taken: {total_time:.2f} seconds")

    # Plot training and validation accuracy
plt.figure(figsize=(10, 5))
plt.plot(train_accuracies, label='Training Accuracy')
plt.plot(val_accuracies, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Regularized CNN Training and Validation Accuracy')
plt.show()

Compare the learning curves before and after adding regularization. Does regularization improve the model's performance on the evaluation data? Is there still evidence of overfitting? At approximately which epoch does the evaluation performance stop improving?

How can we decide when to stop training without knowing the best epoch in advance? Keep in mind that the appropriate stopping epoch may differ depending on the learning rate and model parameters.

Before moving to the next section, think for a moment some logic you can code to help us stop training before the model starts overfitting...
.

.

.

.

.

.

.

Btw, there are many more ways of aiding model training e.g. Data Augmentation, Image Transformation and Normalization, Transfer Learning, Residual Layer etc. which we will cover in the Deep Image Learning module.

## Step 6: Early Stopping and Model Checkpointing

**Early Stopping** Stops training if the validation accuracy does not improve for a specified number of consecutive epochs (patience). It is controlled by a counter (epochs_without_improvement).


**Model Checkpointing**
allows you to save the model’s state whenever it achieves the best performance on the validation set. Uses torch.save() to store the model’s state dictionary to the provided checkpoint_path. Keeps track of the best validation accuracy achieved during training (best_val_accuracy)


**Advantages**

Prevents wasting time on unnecessary epochs once the model stops improving.

Prevents overfitting by breaking off training when dimishing returns on val accuracy.

Checkpointing automatically saves the best model, ensuring you don’t lose progress if training stops early or server connection drops.



For a more advanced implementation, check out Pytorch's Early stopping function: https://pytorch.org/ignite/generated/ignite.handlers.early_stopping.EarlyStopping.html

**solution — stopping and checkpointing**

After each epoch, evaluate on validation data. If the chosen metric improves, save model parameters and reset the patience counter. Otherwise increase the counter; stop when it reaches the threshold. A final test set must remain untouched until the workflow is selected.

### OPTIONAL EXERCISE

**Task 6:** Complete the missing steps in `train_model_with_early_stopping`, including checkpointing and the patience counter.

In [ ]:
def train_model_with_early_stopping(
    model, train_loader, test_loader, criterion, optimizer, device, epochs, patience, checkpoint_path
):
    """
    Train a model with early stopping and model checkpointing.

    Args:
    - model: The PyTorch model to train.
    - train_loader: DataLoader for the training set.
    - test_loader: DataLoader for the validation/test set.
    - criterion: Loss function.
    - optimizer: Optimizer for the model.
    - device: 'cpu' or 'cuda'.
    - epochs: Number of epochs to train.
    - patience: Number of epochs to wait for improvement before stopping.
    - checkpoint_path: Path to save the best model.

    Returns:
    - train_accuracies: List of training accuracies per epoch.
    - val_accuracies: List of validation accuracies per epoch.
    """

    train_accuracies = []
    val_accuracies = []

    best_val_accuracy = 0.0
    epochs_without_improvement = 0

    for epoch in range(epochs):
        # Training phase
        model.train()  # Set the model to training mode
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()  # Clear previous gradients

            outputs = model(images)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            loss.backward()  # Backpropagation
            optimizer.step()  # Update weights

            running_loss += loss.item()  # Accumulate batch loss
            _, predicted = torch.max(outputs, 1)  # Get predicted class
            total += labels.size(0)  # Increment total samples
            correct += (predicted == labels).sum().item()  # Count correct predictions

        # Calculate epoch training accuracy
        train_accuracy = 100 * correct / total
        train_accuracies.append(train_accuracy)

        # Validation phase
        val_loss, val_accuracy = evaluate_model(model, criterion, test_loader, device)
        val_accuracies.append(val_accuracy)

        # Save the model if validation accuracy improves
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            epochs_without_improvement = 0  # Reset the patience counter
            # Save model checkpoint
            print(f"Validation accuracy improved to {val_accuracy:.2f}%. Saving model...")
            torch.save(model.state_dict(), checkpoint_path)
        else:
            epochs_without_improvement += 1

        # Early stopping check
        if epochs_without_improvement >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

        # Print progress
        print(
            f"Epoch [{epoch+1}/{epochs}], "
            f"Train Loss: {running_loss/len(train_loader):.4f}, Train Accuracy: {train_accuracy:.2f}%, "
            f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%"
        )

    print(f"Training completed. Best validation accuracy: {best_val_accuracy:.2f}%.")
    return train_accuracies, val_accuracies


In [ ]:
num_epochs = 50
learning_rate = 0.001

criterion = nn.CrossEntropyLoss()

model = RegularizedCNN().to(device)# send model to gpu
print(f'Using device {device}')
patience = 3
checkpoint_path = "best_model.pth"

optimizer = optim.Adam(model.parameters(), lr=learning_rate,weight_decay=1e-4)


In [ ]:
# Start timer
start_time = time.time()

# Train the model
train_accuracies, val_accuracies = train_model_with_early_stopping(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=num_epochs,
    patience=patience,
    checkpoint_path=checkpoint_path,
)
# Calculate elapsed time
end_time = time.time()
total_time = end_time - start_time

# Print total time and accuracy
print(f"Total time taken: {total_time:.2f} seconds")


## Summary and Analysis
Reflect on your findings:
1. Which activation function performed best? Why do you think that is?
2. How did dropout and L2 regularization impact overfitting?

### Key Takeaways
This practical has shown you how to train nueral networks and how core components like activation functions, regularization, and optimization techniques affect a neural network's performance.


**solution — Summary and Analysis**

1. **Which activation performed best?** Compare the measured validation curves from *your own run*. ReLU often trains more readily than sigmoid in deeper networks because sigmoid derivatives can become small, but there is no universal winner; initialisation, optimiser and data affect the result. A one-epoch capped demo cannot settle this.

2. **How did dropout and L2 regularisation affect overfitting?** Dropout stochastically masks training activations; L2 / weight decay discourages large weights. Look for a smaller train–validation gap and a change in held-out performance, not just lower training accuracy. A claimed numeric gain is run-specific.

**Evaluation caution:** repeated use of `test_loader` for model selection contaminates the final-test interpretation. A proper workflow makes a train/validation split and reserves a separate test set for final evaluation.

**Early stopping:** patience counts epochs without validation improvement. Save a checkpoint at the best validation result and restore it for final testing.